In [ ]:
import jax.numpy as jnp
import numpy as np



def discrete_lqr(Ad, Bd, Q, R, Qf, N):
    """
    Compute K_k matrices for a finite horizon discrete-time LQR problem.

    Parameters:
    A (np.ndarray): Discrete-time system matrix.
    B (np.ndarray): Discrete-time input matrix.
    Q (np.ndarray): Discrete-time State cost matrix.
    R (np.ndarray): Discrete-time Input cost matrix.
    Qf (np.ndarray): Final state cost matrix.
    dt (float): Sampling time.
    N (int): Number of time steps.

    Returns:
    list: A list of K_k matrices.
    """
    # Discretize the cost matrices
    Qd = Q
    Rd = R

    # Initialize the list for K_k matrices
    K_matrices = []
    P_matrices = []
    
    # Initialize P_N
    Pk = Qf
    P_matrices.insert(0, Pk)
    
    # Backward recursion to compute P_k and K_k
    for k in range(N, 0, -1):
        Fk = jnp.linalg.inv(Rd + Bd.T @ Pk @ Bd) @ Bd.T @ Pk @ Ad
        Pk = Fk.T @ Rd @ Fk + (Ad - Bd @ Fk).T @ Pk @ (Ad - Bd @ Fk)
        P_matrices.insert(0, Pk)
        K_matrices.insert(0, Fk)

    
    return K_matrices, P_matrices



def get_GT(states, p, target, n, tau=0.1):
    """
    Given an state, belief, target, current time, and time-discretization, return the analytical solution at that time-step 
    
    states: (8, ) array of current state for both P1 and P2 (x1, y1, vx1, vy1, x2, y2, vx2, vy2)
    p: scalar belief 
    target: 1 for type-1 goal (0, 1) or 0 for type-2 goal (0, -1)
    n: current time-step (backward time)
    tau: time-discretization (default is 0.1)
    
    Returns: u (2, ) array of control input for P1 at time t
             v (2, ) array of control input for P2 at time t
    """

    N = int(1 / tau)  # Number of time steps

    A = jnp.eye(4) + jnp.array([[0, 0, tau, 0], [0, 0, 0, tau], [0, 0, 0, 0], [0, 0, 0, 0]])
    B = jnp.array([[0.5 * tau ** 2, 0], [0, 0.5 * tau ** 2], [tau, 0], [0, tau]])
    Qf = jnp.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]])
    Q = jnp.zeros((4, 4))
    
    R1 = jnp.array([[0.05, 0], [0, 0.025]]) * tau
    R2 = jnp.array([[0.05, 0], [0, 0.1]]) * tau

    K1, P1 = discrete_lqr(A, B, Q, R1, Qf, N)
    K2, P2 = discrete_lqr(A, B, Q, R2, Qf, N)


    tr = N - get_tr(A, B, P1, P2, R1, R2, N=N)  # critical time index

    # print(tr)

    if n <= tr:
        p = 0 if target == 0 else 1
    
    x1 = states[:4]
    x2 = states[4:8]
    goal = jnp.array([0, 2 * p - 1, 0, 0]) * jnp.ones_like(p)
    u = -K1[-n] @ (x1 - goal).T
    v = -K2[-n] @ (x2 - goal).T

    return u, v
    

#use this to get the critical time index
def get_tr(A, B, P1, P2, R1, R2, N=10):
    """
    Return the index of the critical time. 
    """
    def compute_d(A, B, P, R):
        z = jnp.array([[0, 1, 0, 0]]).reshape(-1, 1)
        d = z.T @ A @ P @ B @ jnp.linalg.inv(R) @ B.T @ P @ A @ z
        
        return d

    d1s = jnp.vstack([compute_d(A, B, P1[i], R1) for i in range(N+1)])
    d2s = jnp.vstack([compute_d(A, B, P2[i], R2) for i in range(N+1)])

    f_n = d1s - d2s

    # summation = jnp.array([sum(f_n.reshape(-1, )[:i]) for i in range(N+1)]) 
    # replace with higher order integration
    return np.concatenate(([0.0], np.cumsum(0.5*(f_n[:-1] + f_n[1:])))).argmin()
    


if __name__ == "__main__":
    # Example usage
    states = jnp.array([-0.5, 0, 0, 0, 0.5, 0, 0, 0])
    p = 0.5
    target = 1
    # n = 4  # initial time-step 
    tau = 0.1
    K = int(1/tau)

    for nn in range(0, K): 
        n = K - nn
        u, v = get_GT(states, p, target, n, tau)
        print("Time step:", nn)
        print("Control input for P1:", u)
        print("Control input for P2:", v)

In [4]:
# ---------------------------------------------------------------------------
# Expected game value when get_GT is used *open-loop*
# ---------------------------------------------------------------------------
def analytical_openloop_value(states0, p0=0.5, tau=0.1):
    """
    Compute the expected game value E[L] when the analytical controls are
    generated as an open-loop sequence with fixed (state0, p0).

    Parameters
    ----------
    states0 : jnp.ndarray, shape (8,)
        Initial joint state  (x1,y1,vx1,vy1,  x2,y2,vx2,vy2).
    p0      : float
        Prior probability that the hidden type is 1.
    tau     : float
        Time discretisation (s).

    Returns
    -------
    float  – expected game value under the analytical open-loop policy.
    """
    K_steps = int(1 / tau)

    # ----- local cost matrices (match the game spec) --------------------
    R1 = jnp.array([[0.05, 0.0],
                    [0.0 , 0.025]]) * tau
    R2 = jnp.array([[0.05, 0.0],
                    [0.0 , 0.100]]) * tau
    Kmat = jnp.diag(jnp.array([1., 1., 0., 0.]))

    def value_for_type(target):
        """Simulate one hidden type using the pre-computed open-loop actions."""
        # 1) pre-compute action sequence u_k, v_k (open-loop, no state update)
        seq_u, seq_v = [], []
        for n in range(K_steps, 0, -1):                 # n = K … 1
            u, v = get_GT(states0, p0, target, n, tau)  # uses *initial* state
            seq_u.append(u)
            seq_v.append(v)
        seq_u = jnp.stack(seq_u)    # shape (K,2)
        seq_v = jnp.stack(seq_v)

        # 2) forward simulate dynamics with those actions
        states = states0
        running_cost = 0.0
        for k in range(K_steps):
            u = seq_u[k];  v = seq_v[k]

            # running cost
            running_cost += 0.5 * (u @ R1 @ u - v @ R2 @ v) * tau

            # second-order dynamics
            pos1, vel1 = states[:2],  states[2:4]
            pos2, vel2 = states[4:6], states[6:8]

            pos1 = pos1 + vel1 * tau + 0.5 * u * tau**2
            vel1 = vel1 + u * tau
            pos2 = pos2 + vel2 * tau + 0.5 * v * tau**2
            vel2 = vel2 + v * tau

            states = jnp.concatenate([pos1, vel1, pos2, vel2])

        # terminal cost
        z = jnp.array([0, 1 if target else -1, 0, 0])
        del1 = states[:4]  - z
        del2 = states[4:8] - z
        terminal = 0.5 * (del1.T @ Kmat @ del1 - del2.T @ Kmat @ del2)

        return float(running_cost + terminal)

    # expected value over hidden type
    L0 = value_for_type(target=0)
    L1 = value_for_type(target=1)
    return (1 - p0) * L0 + p0 * L1


# ---------------------------------------------------------------------------
# Quick demo ---------------------------------------------------------------
if __name__ == "__main__":
    init_state = jnp.array([-0.5, 0, 0, 0,   # P1
                            0.5,  0, 0, 0])  # P2
    expected_val = analytical_openloop_value(init_state, p0=0.5, tau=0.1)
    print("Expected game value (open-loop analytical policy):", expected_val)

Expected game value (open-loop analytical policy): -0.20439240336418152
